# DroneFuse - T1: import TSMS-Drone into Kaggle (CPU, Internet On)

TSMS-Drone (Han & Jung, *Scientific Data* 13:407, 2026; figshare doi:10.25452/figshare.plus.30027313, CC BY 4.0)
records four drones and a corner reflector with an FMCW radar, a CW radar and an RF receiver **at the same time**,
hovering 2-30 m away in 2 m steps, 500 measurements per target and distance.

The release is 393 GB of 7z archives, almost all of it raw FMCW maps. This notebook downloads one archive at a time
**directly from figshare into this Kaggle session** (nothing passes through your computer), checks it against
figshare's MD5, keeps only the processed 227 x 227 images of every measurement, packs them into `.tar` files and
deletes the archive before the next one.

**Kaggle settings:** Accelerator = **None** (CPU). Internet = **On**.
**Inputs to add:** `dronefuse-code` (latest). If an earlier part of this import exists, add it too: the archives it
already holds are skipped.
**Output:** `tsms/` with `CW_XXm.tar`, `RF_XXm.tar` (15 each) and `FMCW_XXm_<target>.tar` (75), plus
`import_summary.json` and `t1_report.txt`. Expected size: a few GB.
**Run time:** dominated by the 393 GB download (about 2-6 h, depending on figshare's speed). If the time budget runs
out, the report lists what is missing: save the output, attach it as an input, and run again.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/tsms')
ONLY = None                       # e.g. {'CW', 'RF'} or {2, 4, 6} (FMCW distances in metres); None = everything
TIME_BUDGET_MINUTES = 11 * 60

In [ ]:
import json
import subprocess
import sys

try:
    import py7zr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py7zr'], check=True)
    import py7zr
from dronefuse import environment, tsms
from dronefuse.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'environment': runner.run('measure the machine', environment.collect)}
SCRATCH = environment.pick_scratch_folder(min_free_gb=25) / 'tsms'
earlier = sorted({p.parent for p in INPUT.rglob('*.tar') if p.name.startswith(('CW_', 'RF_', 'FMCW_'))})
print('scratch:', SCRATCH)
print('earlier parts attached:', [str(p) for p in earlier] or 'none')
items = runner.run('read the figshare file list', tsms.archives)
print(f"{len(items or [])} archives, {sum(i['size'] for i in items or []) / 1e9:.0f} GB in total")

In [ ]:
# The import: one archive at a time (download -> MD5 -> images -> tar -> delete).
budget = runner.minutes_left() * 60          # seconds left of the time budget
result = runner.run('import TSMS-Drone', tsms.import_release, OUTPUT, SCRATCH, earlier, ONLY, budget)
summary['import'] = result
with open(OUTPUT / 't1_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)

In [ ]:
# Report
import tarfile

tars = sorted(OUTPUT.glob('*.tar'))
lines = [f"TARS IN THIS OUTPUT: {len(tars)}  ({sum(p.stat().st_size for p in tars) / 1e9:.2f} GB)"]
for sensor in tsms.SENSORS:
    mine = [p for p in tars if p.name.startswith(sensor + '_')]
    images = 0
    for p in mine:
        with tarfile.open(p) as t:
            images += sum(1 for m in t.getmembers() if m.isfile())
    lines.append(f"  {sensor:5s} {len(mine):3d} tars, {images} images")
r = result or {}
lines.append(f"DONE NOW: {len(r.get('done', {}))}   ALREADY IN AN EARLIER PART: {len(r.get('skipped_existing', []))}")
lines.append('NOT REACHED (time budget): ' + (', '.join(r.get('not_reached', [])) or 'none'))
lines.append('FAILED: ' + (json.dumps(r.get('failed')) if r.get('failed') else 'none'))
speeds = [v['mb_per_s'] for v in r.get('done', {}).values()]
if speeds:
    lines.append(f"DOWNLOAD SPEED: median {sorted(speeds)[len(speeds) // 2]} MB/s")
failed = runner.failed()
lines.append('STEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed)))
report = '\n'.join(lines)
(OUTPUT / 't1_report.txt').write_text(report)
print(report)

## After the run

1. Send back `tsms/t1_report.txt`.
2. Open this notebook's saved version -> **Output** -> **New Dataset**, name it **`dronefuse-tsms`** (private).
   If the report lists archives as *not reached*, name this one `dronefuse-tsms-part1`, attach it as an input and
   run again; the second run fetches only what is missing (save it as `dronefuse-tsms-part2`).